## Mental Health Treatment Prediction in Tech Workforce Using Decision Trees and Bagging Ensembles

*1. IMPORT LIBRARIES*

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

  2. *LOAD* *DATASET*

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dmproject/mentalhealth.csv")

print("Shape before cleaning:", df.shape)

Shape before cleaning: (1433, 63)


In [ ]:
def map_condition_category(x):
    if pd.isna(x):
        return "No diagnosis"
    x = x.lower()
    if "anxiety" in x or "panic" in x:
        return "Anxiety-related"
    elif "depression" in x or "bipolar" in x:
        return "Mood-related"
    elif "stress" in x or "burnout" in x:
        return "Stress-related"
    else:
        return "Other"
df['condition_category'] = df['If yes, what condition(s) have you been diagnosed with?'].apply(map_condition_category)


*3. CHOOSE TARGET COLUMN*

In [ ]:
target_col = "Have you ever sought treatment for a mental health issue from a mental health professional?"

# distribution
print(df[target_col].value_counts(dropna=False))

Have you ever sought treatment for a mental health issue from a mental health professional?
1    839
0    594
Name: count, dtype: int64


*4. DATA PREPROCESSING*

In [ ]:
# These columns are with many unique values.They add noise and are hard to encode meaningfully, so we drop them.
maybe_drop = [
    'If maybe, what condition(s) do you believe you have?',
    'If so, what condition(s) were you diagnosed with?'
]

for col in maybe_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

In [ ]:
# Handling Missing Values
print("Missing Values: ", df.isna().sum().sum())

# First, drop rows where the TARGET is missing
df = df.dropna(subset=[target_col])

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Identify categorical vs numeric columns
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# Fill missing numeric with median, categorical with mode
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

print("Any remaining NaNs?", X.isna().sum().sum())

Missing Values:  20127
Any remaining NaNs? 0


In [ ]:
# encoding
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42,stratify=y) #stratify=y so that both sets keep the same Yes/No proportions.


### *MODEL 1: BASELINE DECISION TREE*

In [ ]:
dt_base = DecisionTreeClassifier(random_state=42) #gini

dt_base.fit(X_train, y_train)
y_pred_base = dt_base.predict(X_test)

acc_base = accuracy_score(y_test, y_pred_base)
print("Baseline Decision Tree Accuracy:", acc_base)
print(classification_report(y_test, y_pred_base))

Baseline Decision Tree Accuracy: 0.7979094076655052
              precision    recall  f1-score   support

           0       0.74      0.78      0.76       119
           1       0.84      0.81      0.82       168

    accuracy                           0.80       287
   macro avg       0.79      0.80      0.79       287
weighted avg       0.80      0.80      0.80       287



### *MODEL 2 – Tuned Decision Tree*

In [ ]:
dt_tuned = DecisionTreeClassifier(
    criterion="entropy",   # Use Entropy to measure impurity ,Use Information Gain to decide the best split instead of gini
    max_depth=6,           # limit depth to avoid overfitting
    min_samples_leaf=4,    # minimum samples per leaf node
    random_state=42
)

dt_tuned.fit(X_train, y_train)
y_pred_tuned = dt_tuned.predict(X_test)

acc_tuned = accuracy_score(y_test, y_pred_tuned)
print("Tuned Decision Tree Accuracy:", acc_tuned)
print("Improvement over baseline:", acc_tuned - acc_base)
print(classification_report(y_test, y_pred_tuned))

Tuned Decision Tree Accuracy: 0.8083623693379791
Improvement over baseline: 0.010452961672473893
              precision    recall  f1-score   support

           0       0.75      0.81      0.78       119
           1       0.86      0.81      0.83       168

    accuracy                           0.81       287
   macro avg       0.80      0.81      0.80       287
weighted avg       0.81      0.81      0.81       287



### *MODEL 3 – Bagging (Ensemble of Decision Trees)*

In [ ]:
bag1 = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        criterion="entropy",
        max_depth=6,
        min_samples_leaf=4
    ),
    n_estimators=15,       # number of trees
    max_samples=0.8,       # each tree sees 80% of training data (bootstrap)
    bootstrap=True,
    random_state=42,
    n_jobs=-1              # use all cores
)

bag1.fit(X_train, y_train)
y_pred_bag1 = bag1.predict(X_test)

acc_bag1 = accuracy_score(y_test, y_pred_bag1)
print("Bagging (15 trees) Accuracy:", acc_bag1)
print("Improvement over tuned tree:", acc_bag1 - acc_tuned)
print("Improvement over baseline:", acc_bag1 - acc_base)
print(classification_report(y_test, y_pred_bag1))

Bagging (15 trees) Accuracy: 0.8432055749128919
Improvement over tuned tree: 0.03484320557491283
Improvement over baseline: 0.04529616724738672
              precision    recall  f1-score   support

           0       0.81      0.82      0.81       119
           1       0.87      0.86      0.87       168

    accuracy                           0.84       287
   macro avg       0.84      0.84      0.84       287
weighted avg       0.84      0.84      0.84       287



### *MODEL 4 – Enhanced Bagging (more trees + feature sampling)*

In [ ]:
bag2 = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        criterion="entropy",
        max_depth=None,      # allow deeper trees
        min_samples_leaf=2
    ),
    n_estimators=50,         # more trees
    max_samples=0.9,         # 90% of data per tree
    max_features=0.8,        # each tree sees 80% of features
    bootstrap=True,
    bootstrap_features=True,
    random_state=42,
    n_jobs=-1
)

bag2.fit(X_train, y_train)
y_pred_bag2 = bag2.predict(X_test)

acc_bag2 = accuracy_score(y_test, y_pred_bag2)
print("Enhanced Bagging Accuracy:", acc_bag2)
print("Improvement over baseline:", acc_bag2 - acc_base)
print("Improvement over tuned tree:", acc_bag2 - acc_tuned)
print("Improvement over first bagging model:", acc_bag2 - acc_bag1)
print(classification_report(y_test, y_pred_bag2))

Enhanced Bagging Accuracy: 0.8606271777003485
Improvement over baseline: 0.06271777003484325
Improvement over tuned tree: 0.05226480836236935
Improvement over first bagging model: 0.017421602787456525
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       119
           1       0.88      0.88      0.88       168

    accuracy                           0.86       287
   macro avg       0.86      0.86      0.86       287
weighted avg       0.86      0.86      0.86       287



### *SUMMARY*

In [ ]:
print(" ACCURACY SUMMARY")
print(f"Baseline Decision Tree        : {acc_base:.4f}")
print(f"Tuned Decision Tree           : {acc_tuned:.4f}")
print(f"Bagging (15 trees)            : {acc_bag1:.4f}")
print(f"Enhanced Bagging (50 trees)   : {acc_bag2:.4f}")


 ACCURACY SUMMARY
Baseline Decision Tree        : 0.7979
Tuned Decision Tree           : 0.8084
Bagging (15 trees)            : 0.8432
Enhanced Bagging (50 trees)   : 0.8606


Risk Stratification

In [ ]:
results = X_test.copy()
results['treatment_pred'] = y_pred_bag2
results['actual'] = y_test.values

In [ ]:
def risk_stratify(row):
    score = 0

    # ML prediction
    if row['treatment_pred'] == 1:
        score += 2

    # Family history
    if row.get('Do you have a family history of mental illness?', 0) == 1:
        score += 2

    # Current disorder
    if row.get('Do you currently have a mental health disorder?', 0) == 1:
        score += 3

    # Productivity affected
    if row.get('Do you believe your productivity is ever affected by a mental health issue?', 0) == 1:
        score += 2

    # Condition category weight
    if row.get('condition_category', 0) == 2:   # Mood-related
        score += 2
    elif row.get('condition_category', 0) in [0, 1]:  # Anxiety / Stress
        score += 1

    if score <= 2:
        return "Low Risk"
    elif score <= 5:
        return "Medium Risk"
    else:
        return "High Risk"


In [ ]:
results['risk_level'] = results.apply(risk_stratify, axis=1)


In [ ]:
print(results['risk_level'].value_counts())


risk_level
Medium Risk    175
High Risk       85
Low Risk        27
Name: count, dtype: int64


In [ ]:
def lifestyle_recommendation(risk):
    if risk == "High Risk":
        return "Encourage professional consultation, stress management, physical activity, and use of employer mental health resources."
    elif risk == "Medium Risk":
        return "Recommend work-life balance, mindfulness, awareness programs, and peer support."
    else:
        return "Maintain healthy routines and mental wellness awareness."


In [ ]:
results['recommendation'] = results['risk_level'].apply(lifestyle_recommendation)


In [ ]:
# Take one instance from test set as a "new unseen person"
new_instance = X_test.iloc[[0]]   # double brackets to keep DataFrame
pred_treatment = bag2.predict(new_instance)[0]
if pred_treatment == 1:
    print("Prediction: Likely to seek / need mental health treatment")
else:
    print("Prediction: Unlikely to seek mental health treatment")
instance_with_pred = new_instance.copy()
instance_with_pred['treatment_pred'] = pred_treatment

risk = risk_stratify(instance_with_pred.iloc[0])
print("Risk Level:", risk)
print("Recommendation:", lifestyle_recommendation(risk))


Prediction: Likely to seek / need mental health treatment
Risk Level: Medium Risk
Recommendation: Recommend work-life balance, mindfulness, awareness programs, and peer support.
